# SHAP Explainability Analysis

This notebook visualises feature importance for the trained forecasting model (Revenue + COGS) using SHAP.

**Prerequisites**
- Run `datathon train --mode train-final --model-type lightgbm` (or xgboost) first.
- Run `dbt build` so that `mart_forecast_daily_features` is available.

In [ ]:
import pandas as pd
import seaborn as sns
import shap
from IPython.display import display

from datathon.modeling.recursive import feature_columns
from datathon.modeling.trainer import Trainer
from datathon.utils.data_loaders import load_modeling_data
from datathon.utils.paths import models_dir, warehouse_path

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
shap.initjs()

MODEL_TYPE = "lightgbm"
MODEL_DIR = models_dir() / MODEL_TYPE
WAREHOUSE = warehouse_path()
SAMPLE_SIZE = 500
MAX_DISPLAY = 20

print("Model dir:", MODEL_DIR)

In [ ]:
# Load model artifacts
forecaster, feature_cols, loaded_type = Trainer.load_artifacts(MODEL_DIR)
print(f"Loaded {loaded_type} forecaster with {len(feature_cols)} features.")

In [ ]:
# Load modeling data and sample background distribution
df = load_modeling_data(WAREHOUSE)
cols = feature_columns(df)
X = df[cols].copy()

X_bg = X.sample(n=SAMPLE_SIZE, random_state=42) if len(X) > SAMPLE_SIZE else X

print(f"Background samples: {len(X_bg)}")

## Revenue Model

In [ ]:
explainer_rev = shap.TreeExplainer(forecaster.model_rev)
shap_values_rev = explainer_rev.shap_values(X_bg)

# Handle multi-output wrapper (some SHAP versions return a list for single output)
if isinstance(shap_values_rev, list):
    shap_values_rev = shap_values_rev[0]

# Wrap in Explanation object for plotting
exp_rev = shap.Explanation(
    values=shap_values_rev,
    data=X_bg.values,
    feature_names=cols,
)

shap.plots.beeswarm(exp_rev, max_display=MAX_DISPLAY)

In [ ]:
shap.plots.bar(exp_rev, max_display=MAX_DISPLAY)

## COGS Model

In [ ]:
explainer_cogs = shap.TreeExplainer(forecaster.model_cogs)
shap_values_cogs = explainer_cogs.shap_values(X_bg)

if isinstance(shap_values_cogs, list):
    shap_values_cogs = shap_values_cogs[0]

exp_cogs = shap.Explanation(
    values=shap_values_cogs,
    data=X_bg.values,
    feature_names=cols,
)

shap.plots.beeswarm(exp_cogs, max_display=MAX_DISPLAY)

In [ ]:
shap.plots.bar(exp_cogs, max_display=MAX_DISPLAY)

## Top Features Table

In [ ]:
def top_features_table(shap_values, feature_names, target_name, n=20):
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    mean_abs = pd.Series(
        shap_values.mean(axis=0),
        index=feature_names,
    )
    top = mean_abs.abs().sort_values(ascending=False).head(n).reset_index()
    top.columns = ["feature", "mean_abs_shap"]
    top.insert(0, "rank", range(1, len(top) + 1))
    print(f"\nTop {n} features — {target_name}")
    display(top.style.format({"mean_abs_shap": "{:,.2f}"}).hide(axis="index"))

top_features_table(shap_values_rev, cols, "Revenue", MAX_DISPLAY)
top_features_table(shap_values_cogs, cols, "COGS", MAX_DISPLAY)